<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/7skaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_skaner.py

from __future__ import annotations

import json
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import (
    Any,
    Callable,
    Final,
    Protocol,
)

from modul_filtry import (
    AdvancedMomentumFilter,
    FiltrMomentum,
    FiltrSMA,
    FiltrZmiennosci,
    SpolkaDoFiltrow,
    StatusFiltra,
    WynikFiltra,
)


FOLDER_PROJEKTU: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)


FiltrCallable = Callable[
    [SpolkaDoFiltrow],
    WynikFiltra
]

ObserverCallable = Callable[
    [str],
    None
]


class ScannerRepository(Protocol):

    def pobierz_spolke(
        self,
        ticker: str,
    ) -> SpolkaDoFiltrow:
        ...


class ScannerResultRepository(Protocol):

    def zapisz_wynik(
        self,
        wynik: WynikSkanera,
    ) -> Path:
        ...


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True,
)
class WynikSkanera:
    ticker: str

    przeszedl: bool

    wyniki_filtrow: list[WynikFiltra] = field(
        default_factory=list
    )

    liczba_filtrow: int = field(
        init=False
    )

    liczba_passed: int = field(
        init=False
    )

    liczba_failed: int = field(
        init=False
    )

    timestamp: datetime = field(
        default_factory=datetime.now,
        compare=False,
        repr=False
    )

    def __post_init__(self) -> None:

        object.__setattr__(
            self,
            "liczba_filtrow",
            len(self.wyniki_filtrow)
        )

        passed: int = sum(
            1
            for wynik in self.wyniki_filtrow
            if wynik.status
            == StatusFiltra.PASSED
        )

        failed: int = sum(
            1
            for wynik in self.wyniki_filtrow
            if wynik.status
            == StatusFiltra.FAILED
        )

        object.__setattr__(
            self,
            "liczba_passed",
            passed
        )

        object.__setattr__(
            self,
            "liczba_failed",
            failed
        )


class JsonScannerRepository:

    def __init__(
        self,
        folder: Path,
    ) -> None:

        self.folder = folder

    def pobierz_spolke(
        self,
        ticker: str,
    ) -> SpolkaDoFiltrow:

        ticker = (
            ticker
            .strip()
            .upper()
        )

        plik: Path = (
            self.folder
            / f"{ticker}_filtry.json"
        )

        if not plik.exists():
            raise FileNotFoundError(
                f"brak pliku z modulu 6: "
                f"{plik}"
            )

        with open(
            plik,
            "r",
            encoding="utf-8",
        ) as f:

            dane: Any = json.load(
                f
            )

        if not isinstance(
            dane,
            dict,
        ):
            raise ValueError(
                "plik modulu 6 "
                "musi zawierac dict"
            )

        if "dane_wejsciowe" not in dane:
            raise ValueError(
                "brak dane_wejsciowe "
                "w pliku modulu 6"
            )

        dane_modulu_5: Any = (
            dane["dane_wejsciowe"]
        )

        if not isinstance(
            dane_modulu_5,
            dict,
        ):
            raise ValueError(
                "dane_wejsciowe "
                "musza byc dict"
            )

        if "obliczenia" not in dane_modulu_5:
            raise ValueError(
                "brak sekcji obliczenia"
            )

        obliczenia: dict[str, Any] = (
            dane_modulu_5[
                "obliczenia"
            ]
        )

        if "zakres_cen" not in obliczenia:
            raise ValueError(
                "brak zakres_cen"
            )

        zakres: dict[str, Any] = (
            obliczenia[
                "zakres_cen"
            ]
        )

        return SpolkaDoFiltrow(
            ticker=ticker,

            srednia_kroczaca=float(
                obliczenia[
                    "srednia_kroczaca_20"
                ]
            ),

            stopa_zwrotu=float(
                obliczenia[
                    "stopa_zwrotu_proc"
                ]
            ),

            sredni_wolumen=float(
                obliczenia[
                    "sredni_wolumen_20"
                ]
            ),

            zmiennosc=float(
                obliczenia[
                    "zmiennosc_proc"
                ]
            ),

            liczba_notowan=int(
                obliczenia[
                    "liczba_notowan"
                ]
            ),

            cena_minimum=float(
                zakres[
                    "minimum"
                ]
            ),

            cena_maksimum=float(
                zakres[
                    "maksimum"
                ]
            ),

            cena_srednia=float(
                zakres[
                    "srednia"
                ]
            ),
        )


class JsonScannerResultRepository:

    def __init__(
        self,
        folder: Path,
    ) -> None:

        self.folder = folder

    def zapisz_wynik(
        self,
        wynik: WynikSkanera,
    ) -> Path:

        self.folder.mkdir(
            parents=True,
            exist_ok=True,
        )

        plik: Path = (
            self.folder
            / f"{wynik.ticker}_skaner.json"
        )

        dane: dict[str, Any] = {
            "ticker":
                wynik.ticker,

            "przeszedl":
                wynik.przeszedl,

            "podsumowanie": {
                "liczba_filtrow":
                    wynik.liczba_filtrow,

                "liczba_passed":
                    wynik.liczba_passed,

                "liczba_failed":
                    wynik.liczba_failed,
            },

            "wyniki_filtrow": [
                {
                    "typ":
                        filtr.typ_filtra.value,

                    "status":
                        filtr.status.value,

                    "wartosc":
                        filtr.wartosc,

                    "prog":
                        filtr.prog,

                    "opis":
                        filtr.opis,
                }

                for filtr
                in wynik.wyniki_filtrow
            ],

            "timestamp":
                wynik.timestamp.isoformat(),
        }

        with open(
            plik,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                dane,
                f,
                ensure_ascii=False,
                indent=2,
            )

        return plik


class Skaner:

    def __init__(
        self,
        repository: ScannerRepository,
        filtry: list[FiltrCallable],
        observers: list[
            ObserverCallable
        ] | None = None,
    ) -> None:

        self.repository = repository

        self.filtry = filtry

        self.observers = (
            list(observers)
            if observers is not None
            else []
        )

    def dodaj_observer(
        self,
        observer: ObserverCallable,
    ) -> None:

        self.observers.append(
            observer
        )

    def notify(
        self,
        komunikat: str,
    ) -> None:

        for observer in self.observers:

            observer(
                komunikat
            )

    def pobierz_spolke(
        self,
        ticker: str,
    ) -> SpolkaDoFiltrow:

        self.notify(
            f"pobieranie spolki {ticker}"
        )

        return (
            self.repository
            .pobierz_spolke(
                ticker
            )
        )

    def uruchom_filtry(
        self,
        spolka: SpolkaDoFiltrow,
    ) -> list[WynikFiltra]:

        wyniki: list[
            WynikFiltra
        ] = []

        for filtr in self.filtry:

            self.notify(
                f"uruchamianie filtra "
                f"{type(filtr).__name__} "
                f"dla {spolka.ticker}"
            )

            wynik: WynikFiltra = (
                filtr(
                    spolka
                )
            )

            wyniki.append(
                wynik
            )

            self.notify(
                f"{spolka.ticker}: "
                f"{wynik.typ_filtra.value} "
                f"-> {wynik.status.value}"
            )

        return wyniki

    def podejmij_decyzje(
        self,
        wyniki: list[WynikFiltra],
    ) -> bool:

        if not wyniki:
            raise ValueError(
                "brak wynikow filtrow"
            )

        return all(
            wynik.status
            == StatusFiltra.PASSED

            for wynik in wyniki
        )

    def skanuj_spolke(
        self,
        ticker: str,
    ) -> WynikSkanera:

        self.notify(
            f"start skanowania {ticker}"
        )

        spolka: SpolkaDoFiltrow = (
            self.pobierz_spolke(
                ticker
            )
        )

        wyniki: list[
            WynikFiltra
        ] = self.uruchom_filtry(
            spolka
        )

        decyzja: bool = (
            self.podejmij_decyzje(
                wyniki
            )
        )

        wynik = WynikSkanera(
            ticker=spolka.ticker,
            przeszedl=decyzja,
            wyniki_filtrow=wyniki,
        )

        self.notify(
            f"koniec skanowania "
            f"{ticker}: "
            f"{'PASSED' if decyzja else 'FAILED'}"
        )

        return wynik

    def skanuj(
        self,
        tickery: list[str],
    ) -> list[WynikSkanera]:

        wyniki: list[
            WynikSkanera
        ] = []

        for ticker in tickery:

            wynik: WynikSkanera = (
                self.skanuj_spolke(
                    ticker
                )
            )

            wyniki.append(
                wynik
            )

        return wyniki

    def spolki_ktore_przeszly(
        self,
        wyniki: list[WynikSkanera],
    ) -> list[WynikSkanera]:

        return [
            wynik
            for wynik in wyniki
            if wynik.przeszedl
        ]


class ConsoleObserver:

    def notify(
        self,
        komunikat: str,
    ) -> None:

        print(
            "[SKANER]",
            komunikat
        )


class ScannerFacade:

    def __init__(
        self,
        skaner: Skaner,
        result_repository:
            ScannerResultRepository,
    ) -> None:

        self.skaner = skaner

        self.result_repository = (
            result_repository
        )

    def wykonaj(
        self,
        tickery: list[str],
    ) -> list[WynikSkanera]:

        wyniki: list[
            WynikSkanera
        ] = self.skaner.skanuj(
            tickery
        )

        for wynik in wyniki:

            self.result_repository.zapisz_wynik(
                wynik
            )

        return wyniki


def run() -> None:

    tekst: str = input(
        "podaj ticker lub tickery "
        "oddzielone przecinkami: "
    )

    tickery: list[str] = [
        ticker.strip().upper()

        for ticker
        in tekst.split(",")

        if ticker.strip()
    ]

    if not tickery:
        raise ValueError(
            "nie podano zadnego tickera"
        )

    repository = (
        JsonScannerRepository(
            folder=FOLDER_PROJEKTU
        )
    )

    result_repository = (
        JsonScannerResultRepository(
            folder=FOLDER_PROJEKTU
        )
    )

    filtr_sma = FiltrSMA(
        minimalna_relacja=1.0
    )

    filtr_momentum = FiltrMomentum(
        minimalne_momentum=0.0
    )

    filtr_advanced = (
        AdvancedMomentumFilter(
            minimalne_momentum=2.0,
            minimalny_wolumen=100_000.0,
        )
    )

    filtr_zmiennosci = (
        FiltrZmiennosci(
            maksymalna_zmiennosc=5.0
        )
    )

    filtry: list[
        FiltrCallable
    ] = [
        filtr_sma,
        filtr_momentum,
        filtr_advanced,
        filtr_zmiennosci,
    ]

    observer = ConsoleObserver()

    observer_callable: ObserverCallable = (
        observer.notify
    )

    skaner = Skaner(
        repository=repository,
        filtry=filtry,
        observers=[
            observer_callable
        ],
    )

    facade = ScannerFacade(
        skaner=skaner,
        result_repository=(
            result_repository
        ),
    )

    wyniki: list[
        WynikSkanera
    ] = facade.wykonaj(
        tickery
    )

    print(
        "\nPODSUMOWANIE SKANERA"
    )

    for wynik in wyniki:

        print(
            wynik.ticker,
            "->",
            (
                "PASSED"
                if wynik.przeszedl
                else "FAILED"
            ),
            "|",
            wynik.liczba_passed,
            "/",
            wynik.liczba_filtrow,
            "filtrow"
        )

    przeszly: list[
        WynikSkanera
    ] = (
        skaner
        .spolki_ktore_przeszly(
            wyniki
        )
    )

    print(
        "\nSPOLKI, KTORE PRZESZLY:"
    )

    if not przeszly:

        print(
            "brak"
        )

    else:

        for wynik in przeszly:

            print(
                "-",
                wynik.ticker
            )

    print(
        "\nzapisano wyniki:"
    )

    for wynik in wyniki:

        print(
            FOLDER_PROJEKTU
            / f"{wynik.ticker}_skaner.json"
        )

    print(
        "\nMODUL SKANERA "
        "DZIALA POPRAWNIE"
    )